# POSCAR to LAMMPS conversion

Convert one directory of POSCAR files into LAMMPS data files.

The conversion logic is handled by the internal `lammps_md_preparation` package under `lammps_md_preparation/python/`, so the notebook only carries configuration and reporting.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "python" / "lammps_md_preparation" / "__init__.py").exists():
    NOTEBOOK_DIR = cwd
    package_root = cwd / "python"
elif (cwd / "lammps_md_preparation" / "python" / "lammps_md_preparation" / "__init__.py").exists():
    NOTEBOOK_DIR = cwd / "lammps_md_preparation"
    package_root = NOTEBOOK_DIR / "python"
else:
    raise RuntimeError("Run this notebook from the repository root or from lammps_md_preparation/.")

if str(package_root) not in sys.path:
    sys.path.insert(0, str(package_root))

from pprint import pprint
from lammps_md_preparation.utils import collect_poscar_files, sanitize_basename, specorder_from_atoms, write_lammps_data_file
from ase.io import read


In [ ]:
# Configuration
SRC_DIR = NOTEBOOK_DIR / "input" / "poscars"
POSCAR_PATTERN = "POSCAR*"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "lammps_inputs"
ONE_DIR_PER_POSCAR = True
ATOM_STYLE = "atomic"
UNITS = "metal"


In [ ]:
files_poscar = collect_poscar_files(SRC_DIR, POSCAR_PATTERN)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Found {len(files_poscar)} POSCAR files in {SRC_DIR}")

report = []
for poscar_path in files_poscar:
    atoms = read(str(poscar_path), format="vasp")
    specorder = specorder_from_atoms(atoms)
    target_dir = OUTPUT_DIR / sanitize_basename(poscar_path) if ONE_DIR_PER_POSCAR else OUTPUT_DIR
    target_dir.mkdir(parents=True, exist_ok=True)
    output_path = target_dir / "lmp_geo.lmp"
    write_lammps_data_file(
        atoms,
        output_path,
        specorder=specorder,
        atom_style=ATOM_STYLE,
        units=UNITS,
    )
    report.append({
        "poscar": poscar_path.name,
        "output": str(output_path),
        "specorder": specorder,
        "natoms": len(atoms),
    })

pprint(report)
